<a href="https://colab.research.google.com/github/jhuangjennifer/573ChineseEnglishSummarization/blob/jjnhuang-work/notebooks/inference_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers

In [ ]:
# Load summarization model directly
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

bart_tokenizer = AutoTokenizer.from_pretrained("yunu919/bart-large-dialogue-summarization")
bart_model = AutoModelForSeq2SeqLM.from_pretrained("yunu919/bart-large-dialogue-summarization")

mbart_tokenizer = AutoTokenizer.from_pretrained("yunu919/mbart-large-dialogue-summarization")
mbart_model = AutoModelForSeq2SeqLM.from_pretrained("yunu919/mbart-large-dialogue-summarization")

In [ ]:
# Initialize translation tokenizer and model: https://huggingface.co/Helsinki-NLP/opus-mt-en-zhhttps://huggingface.co/Helsinki-NLP/opus-mt-en-zh

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

translate_tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-zh")
translate_model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-zh")

In [ ]:
# Load XSAMSum dataset
from datasets import load_dataset
import os

BASE_DIR = f"sample_data"

data_files = {
    "test": os.path.join(BASE_DIR, "test.json"),
}

dataset_dict = load_dataset("json", data_files=data_files)

In [ ]:
import os
import torch
from tqdm.auto import tqdm

def summarize_dialogue(text, summary_tokenizer, summary_model):
    inputs = summary_tokenizer(
        text, return_tensors="pt", max_length=512, truncation=True
    ).to(summary_model.device)

    with torch.no_grad():
        summary_ids = summary_model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=150,
            num_beams=4,
            early_stopping=True
        )

    # Intermediate English summary
    predicted_en = summary_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    # Translate into Chinese
    tokenized_summary = translate_tokenizer(
        predicted_en, return_tensors="pt"
    ).to(translate_model.device)

    with torch.no_grad():
        output = translate_model.generate(**tokenized_summary)

    predicted_zh = translate_tokenizer.decode(output[0], skip_special_tokens=True)

    return predicted_en, predicted_zh


# Set this to BART or mBART
tokenizer = bart_tokenizer   # or bart_tokenizer
model = bart_model           # or bart_model

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

output_en_filepath = os.path.join(output_dir, "bart_predictions_en.txt")
output_zh_filepath = os.path.join(output_dir, "bart_predictions_zh.txt")

test_dialogues = dataset_dict["test"]["dialogue"]
total_samples = len(test_dialogues)

with open(output_en_filepath, "w", encoding="utf-8") as f_en, \
     open(output_zh_filepath, "w", encoding="utf-8") as f_zh:

    pbar = tqdm(total=total_samples, desc="Running inference")

    for i, text in enumerate(test_dialogues):
        predicted_en, predicted_zh = summarize_dialogue(text, tokenizer, model)

        f_en.write(predicted_en + "\n")
        f_zh.write(predicted_zh + "\n")

        remaining = total_samples - (i + 1)
        pbar.set_postfix(done=i + 1, remaining=remaining)
        pbar.update(1)

    pbar.close()

print(f"Done. Saved to: {output_en_filepath} and {output_zh_filepath}")

In [ ]:
from google.colab import output
output.clear()